In [138]:
### Libaries - Joe
from smo import smo #compiled python from ipynb from assignment 5 code
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [139]:
### Random Seed & Other Constants - Joe
RANDOM_SEED = 0
EPS = 10**-8
TAU = 10**-3
MAX_ITER = 10**4

### Implement MLP - Multilayer Perceptron - Joe

In [140]:
### Sigmoid activation function from previous hw - Joe
def sigmoid(z):
    # no overflow warning for z=abs(1000)
    # clip to abs(500), beyond that call np.exp()
    z = np.clip(z, -500,500)
    return 1 / (1 + np.exp(-z))

In [141]:
### Define a class to abstract the weights and biases of the mlp - Joe
class MLP_W:
    def __init__(self):
        # Weights and biases to be init later
        self.Hidden_Weights = None
        self.Hidden_Bias = None
        self.Output_Weights = None
        self.Output_Bias = None

        # alpha momentum for weights as suggested in slide 67
        self.Hidden_W_Alpha = None
        self.Hidden_B_Alpha = None
        self.Output_W_Alpha = None
        self.Output_B_Alpha = None

In [142]:
### Similar to SMO approach - helper function to init MLP - Joe
def initMLP(H, RNG):
    W = MLP_W()
    W.Hidden_Weights = RNG.uniform(-0.5, 0.5,size=(2,H)) # two inputs
    W.Hidden_Bias =RNG.uniform(-0.5, 0.5,size=(1,H))
    W.Output_Weights = RNG.uniform(-0.5, 0.5,size=(H,1)) # one output
    W.Output_Bias =RNG.uniform(-0.5, 0.5,size=(1,1))

    W.Hidden_W_Alpha = np.zeros_like(W.Hidden_Weights)
    W.Hidden_B_Alpha = np.zeros_like(W.Hidden_Bias)
    W.Output_W_Alpha = np.zeros_like(W.Output_Weights)
    W.Output_B_Alpha = np.zeros_like(W.Output_Bias)
    return W

In [143]:
### Implement forward - calculate hidden&output activation terms for each neuron - Joe
def forward(X,W):
    hidden = sigmoid(X @ W.Hidden_Weights + W.Hidden_Bias)
    output = sigmoid(hidden @ W.Output_Weights + W.Output_Bias)
    return output,hidden

In [144]:
### Implement backward - calculates output&hidden error terms for each neuron - Joe
def backward(W, output,hidden,t):
    # taking from delta_o = o(1-o)(t-o), 
    # where t= 0.9 if y=+1 then 0.1 otherwise y=-1
    delta_o = output * (1-output)* (t-output)
    # hidden 
    delta_h = hidden * (1-hidden) *(delta_o @ W.Output_Weights.T)
    return delta_o, delta_h

In [145]:
### Implement - update weights by computing deltas for each neuron - Joe
def updateWeights(X,W, hidden, delta_o, delta_h, eta,alpha):
    # use momentum and calculate deltas
    delta_Output_Weights = eta*(hidden.T @ delta_o) +alpha*W.Output_W_Alpha
    delta_Output_Bias = eta * delta_o + alpha * W.Output_B_Alpha

    delta_Hidden_Weights = eta * (X.T @ delta_h) + alpha * W.Hidden_W_Alpha
    delta_Hidden_Bias = eta * delta_h + alpha * W.Hidden_B_Alpha

    # apply updates
    W.Output_Weights += delta_Output_Weights
    W.Output_Bias += delta_Output_Bias
    W.Hidden_Weights += delta_Hidden_Weights
    W.Hidden_Bias += delta_Hidden_Bias

    # save momentum
    W.Output_W_Alpha = delta_Output_Weights
    W.Output_B_Alpha = delta_Output_Bias
    W.Hidden_W_Alpha = delta_Hidden_Weights
    W.Hidden_B_Alpha = delta_Hidden_Bias
    
    return W

In [ ]:
### Implement m_tlprain - Joe
def mlp_train(X,y,H,eta=0.3,alpha=0.3,epochs=1000,seed=RANDOM_SEED):
    #https://numpy.org/doc/stable/reference/random/generator.htmlNe
    # initialize multilayer perceptron
    RNG = np.random.default_rng(seed)
    W = initMLP(H, RNG)

    # target for this assignment is 0.9
    targets = np.where(y==1, 0.9,0.1)
    epochMonitor = {
        "errs" : [],
        "miscs" : []
    }

    # training loop
    for epoch in range(epochs):

        # shuffle training examples
        idxs = RNG.permutation(len(X))

        # stochastic updates
        for i in idxs:
            # obtain example and label
            ex = X[i:i+1] # for shape (2,1)
            t = targets[i]

            # update mlp
            output, hidden = forward(ex, W)
            delta_o,delta_h = backward(W, output, hidden, t)
            W = updateWeights(ex, W, hidden, delta_o, delta_h, eta, alpha)

        # evaluate for epoch monitor
        outputs, hiddens = forward(X,W)
        outputs = outputs.ravel()

        # mse Ed[⃗w] ≡ 1/2(td − od)2
        err = 0.5 * np.mean(targets-outputs)**2

        # miscs as a rate
        preds = np.where(outputs >0.5,1,-1)
        misc = np.mean(preds !=y)

        epochMonitor["errs"].append(err)
        epochMonitor["miscs"].append(misc)
    return W, epochMonitor

### Data Sets & Experiments

In [153]:
### Helper standardize from previous hw - Joe
def standardize(Xtr, Xte):
    """Mean/std from the TRAINING set only, applied to both sets."""
    mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
    return (Xtr - mu) / sd, (Xte - mu) / sd, mu, sd

In [148]:
### Helper stratified split from previous hw - Joe
def stratified_split(X, y,test_fraction=0.2, seed=RANDOM_SEED):
    """Seeded, class-proportional train/test split."""
    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for k in np.unique(y):
        idx_k = np.where(y == k)[0]
        rng.shuffle(idx_k)
        n_test_k = int(round(len(idx_k) * test_fraction))
        test_idx.extend(idx_k[:n_test_k])
        train_idx.extend(idx_k[n_test_k:])
    train_idx, test_idx = np.array(train_idx), np.array(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [149]:
### Data set 1 - Iris,Setosa vs Versicolor - Joe
trainDF = pd.read_csv("data/iris_setosa_versicolor_train.csv")
testDF = pd.read_csv("data/iris_setosa_versicolor_test.csv")

# split into train and test, convert to numpy for mlp
Xtr = np.asarray(trainDF[["sepal_length","petal_length"]].values, dtype=float)
Xte = np.asarray(testDF[["sepal_length","petal_length"]].values, dtype=float)
ytr = np.asarray(trainDF["y"].values).ravel()
yte = np.asarray(testDF["y"].values).ravel()

In [150]:
### Data set 2 - XOR - Joe
trainDF = pd.read_csv("data/xor_train.csv")
testDF = pd.read_csv("data/xor_test.csv")

# split into train and test
Xtr = trainDF[["x1","x2"]].values
Xte = testDF[["x1","x2"]].values
ytr = trainDF["y"].values
yte = testDF["y"].values

In [151]:
### Data set 3 - RINGS - Joe
trainDF = pd.read_csv("data/rings_train.csv")
testDF = pd.read_csv("data/rings_test.csv")

# split into train and test
Xtr = trainDF[["x1","x2"]].values
Xte = testDF[["x1","x2"]].values
ytr = trainDF["y"].values
yte = testDF["y"].values

In [155]:
### Data set 4 - iris_versicolor_virginica - Joe
dataDF = pd.read_csv("data/iris_versicolor_virginica.csv")

X = dataDF[["petal_length", "petal_width"]].values
y = dataDF["y"].values

# 60/20/20
Xtr, Xte, ytr, yte = stratified_split(X, y, test_fraction=1/5)
Xtr, Xval, ytr, yval = stratified_split(Xtr, ytr, test_fraction=1/4)

# standardize
mu = Xtr.mean(axis=0)
std = Xtr.std(axis=0)
std[std == 0] = 1

Xtr = (Xtr - mu) / std
Xval = (Xval - mu) / std
Xte = (Xte - mu) / std

print(len(Xtr), len(Xval), len(Xte))

60 20 20
